# Curvature-Coupled Dark Energy: Expansion, Distance, and CMB Figures

This notebook reproduces three observational figures used in the
**Curvature-Coupled Dark Energy (CCDE)** paper:

- **Fig. 3:** conformal expansion rate compared with cosmic-chronometer measurements,
- **Fig. 4:** Type Ia supernova magnitude--redshift relation,
- **Fig. 8:** CMB temperature angular power spectrum compared with Planck 2018 data.

The numerical calculations and data structures follow the original analysis.


In [ ]:

import os
from os.path import exists
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset


def nested_dict(n, value_type):
    """Create an n-level nested defaultdict."""
    if n == 1:
        return defaultdict(value_type)
    return defaultdict(lambda: nested_dict(n - 1, value_type))


data = nested_dict(5, list)
out_arr = nested_dict(5, list)


def parse_bash(sigma_str):
    """Convert a whitespace-separated tuple-like string to numeric values."""
    cleaned_str = sigma_str.strip('()').replace('"', '')
    sigma_list = cleaned_str.split()

    def convert_value(value):
        num = float(value)
        return int(num) if num == int(num) else num

    return np.array([convert_value(value) for value in sigma_list], dtype=object)


## 1. Load CCDE background outputs

Load the `hi_class` background tables for the $(\sigma,\alpha)$ parameter grid together
with the reference $\Lambda$CDM background. These background outputs are used for
paper Figs. 3 and 4.


In [ ]:
# -------------------------
# Configuration
# -------------------------
redshifts = [100, 50, 30, 20, 10, 6, 5, 4, 3, 2.5, 2, 1.75, 1.5, 1.25, 1.1, 1.0,
             0.9, 0.8, 0.75, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.1, 0.0]

data_address = "./../DataGenerator/transfer_functions/"
load_data = True

# Use strings so directory names match exactly
alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]  


# -------------------------
# Build alpha/sigma arrays
# -------------------------
alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

# Run labels indexed by (sigma, alpha)
out_arr = np.empty((len(sigma), len(alpha)), dtype=object)

# Initialize nested entries
data.setdefault('bg_data', {})
for s in sigma:
    data['bg_data'].setdefault(f'sigma={s}', {})
    for a in alpha:
        data['bg_data'][f'sigma={s}'].setdefault(f'alpha={a}', None)

# -------------------------
# Load LCDM as (alpha=0, sigma=0)
# -------------------------
lcdm_dir = os.path.join(data_address, "LCDM")
lcdm_bg = os.path.join(lcdm_dir, "LCDM_background.dat")

data['bg_data'].setdefault('sigma=0.0', {})
data['bg_data']['sigma=0.0'].setdefault('alpha=0.0', None)

if exists(lcdm_bg):
    print("\033[94mLoading LCDM as alpha=0, sigma=0\033[0m")
    data['bg_data']['sigma=0.0']['alpha=0.0'] = np.loadtxt(lcdm_bg)
else:
    print(lcdm_bg,"\033[91mLCDM files not found!\033[0m")

# -------------------------
# Load CCDE runs (alphas × sigmas grid)
# -------------------------
total_loaded = 0
missing = []

if load_data:
    for a_str in alpha:
        for s_str in sigma:
            output_dir = f"sigma{s_str}_alpha{a_str}"
            base_path  = os.path.join(data_address, "run_"+output_dir)
            bg_file    = os.path.join(base_path, "CCDE_"+output_dir+"_background.dat")

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i][j] = output_dir

            if exists(bg_file):
                print(f"\033[94m{output_dir} exists — loading\033[0m")
                data['bg_data'][f'sigma={s_str}'][f'alpha={a_str}'] = np.loadtxt(bg_file)
                total_loaded += 1
            else:
                missing.append(bg_file)

print("Number of CCDE simulations loaded:", total_loaded)
print("Alphas loaded:", list(alpha))
print("Sigmas loaded:", list(sigma))
if missing:
    print(f"\033[93mMissing {len(missing)} files (showing up to 10):\033[0m")
    for m in missing[:10]:
        print("  -", m)

data_obs = np.genfromtxt("Data/CCHz.txt", dtype=None, names=True, encoding=None)


## 2. Conformal expansion rate — paper Fig. 3

The `hi_class` background table stores the physical Hubble rate $H(z)$ in units of
$\mathrm{Mpc}^{-1}$. After converting it to $\mathrm{km\,s^{-1}\,Mpc^{-1}}$, we plot
the conformal Hubble parameter

$\mathcal{H}(z)=aH(z)=\frac{H(z)}{1+z}.$
The cosmic-chronometer measurements are converted in the same way. The lower panel
shows $\mathcal{H}/\mathcal{H}_{\Lambda\mathrm{CDM}}$.


In [ ]:

c = 299792.458  # speed of light [km/s]

# ---- Plot style parameters ----
text_size = 30
fig_size_x = 12
fig_size_y = 10
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# ---- Figure: expansion history and ratio to LambdaCDM ----
fig = plt.figure(figsize=(fig_size_x, fig_size_y))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)

ax = fig.add_subplot(gs[0])
ax_rat = fig.add_subplot(gs[1], sharex=ax)

# Fixed paper-wide model-to-colour mapping.
# The first entry is reserved for the LambdaCDM reference.
colors = [
    '#000000',  # LambdaCDM
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = ["0.0", "0.001", "0.1", "0.001", "-0.1",
                "0.1", "-0.1", "0.1", "0.001", "-0.05"]
sigma_consts = ["0.0", "0.001", "0.3", "0.3", "0.3",
                "1.", "1.", "1.5", "1.5", "1.5"]

COL_Z = 0
COL_H = 3


def load_sorted_H(arr):
    """Return redshift and physical H(z) sorted by increasing redshift."""
    z = arr[:, COL_Z]
    order = np.argsort(z)
    z = z[order]
    H_phys = c * arr[order, COL_H]
    return z, H_phys


# ---- LambdaCDM baseline ----
lcdm_arr = data['bg_data']['sigma=0.0']['alpha=0.0']
z_LCDM, H_LCDM = load_sorted_H(lcdm_arr)

# Conformal Hubble parameter: H_conf = a H_phys = H_phys/(1+z).
Hconf_LCDM = H_LCDM / (1.0 + z_LCDM)

# Interpolate in log(1+z), which remains finite at z=0.
interp_LCDM = interp1d(
    np.log1p(z_LCDM),
    Hconf_LCDM,
    kind='linear',
    bounds_error=False,
    fill_value="extrapolate"
)

# ---- CCDE models ----
for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    arr = data['bg_data'].get(sigma_key, {}).get(alpha_key, None)
    if arr is None:
        continue

    z_model, H_model = load_sorted_H(arr)
    Hconf = H_model / (1.0 + z_model)

    color = colors[num]

    if sigma_l == "0.0" and alpha_l == "0.0":
        H_LCDM_eval = interp_LCDM(np.log1p(z_model))
        ratio = Hconf / H_LCDM_eval
        ax_rat.plot(z_model, ratio, ':', lw=lw_f, c=color)
        continue

    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'

    ax.plot(
        z_model, Hconf, '-',
        lw=lw_f, c=color, label=label
    )

    H_LCDM_eval = interp_LCDM(np.log1p(z_model))
    ratio = Hconf / H_LCDM_eval

    ax_rat.plot(
        z_model, ratio, '-',
        lw=lw_f, c=color
    )

# ---- Cosmic-chronometer measurements ----
z_data = data_obs["z"]
H_data = data_obs["H"]
dH_data = data_obs["dH"]

# Convert the measured physical H(z) to the conformal Hubble parameter.
# The first entry is omitted because the figure uses a logarithmic redshift axis.
ax.errorbar(
    z_data[1:],
    H_data[1:] / (1.0 + z_data[1:]),
    yerr=dH_data[1:] / (1.0 + z_data[1:]),
    fmt='o',
    ms=7,
    elinewidth=2.0,
    capsize=4,
    color='k',
    mfc='white',
    mec='k',
    label=r'Cosmic chronometer $\mathcal{H}(z)$'
)

# ---- Formatting: upper panel ----
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.01, 10)
ax.set_ylim(10, 150)
ax.set_ylabel(r'$\mathcal{H}(z)\;[\mathrm{km\,s^{-1}\,Mpc^{-1}}]$')
ax.grid(True, which='both', alpha=0.3)
ax.tick_params(direction='in', top=True, right=True)
plt.setp(ax.get_xticklabels(), visible=False)

# ---- Formatting: ratio panel ----
ax_rat.set_xscale('log')
ax_rat.set_xlim(0.01, 10)
ax_rat.set_ylim(0.93, 1.36)
ax_rat.set_ylabel(r'$\mathcal{H}/\mathcal{H}_{\Lambda\mathrm{CDM}}$')
ax_rat.set_xlabel(r'$z$')
ax_rat.axhline(1.0, color='k', lw=1.5)

ax_rat.yaxis.set_major_locator(MultipleLocator(0.1))
ax_rat.yaxis.set_minor_locator(MultipleLocator(0.05))
ax_rat.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

ax_rat.grid(True, which='both', alpha=0.3)
ax_rat.tick_params(direction='in', top=True, right=True)

ax.legend(frameon=True, fontsize=16, loc='best', ncol=2)

# plt.tight_layout()

plt.savefig(
    './Figs/Hubble_cosmic_chronometer.pdf',
    format='pdf', dpi=300,
    bbox_inches='tight', pad_inches=0.1
)

plt.show()


## 3. Type Ia supernova magnitude--redshift relation — paper Fig. 4

Using the luminosity distance $D_L(z)$ from the background solution, the theoretical
apparent magnitude is constructed as
\[
m_B(z)=5\log_{10}\!\left[D_L(z)/\mathrm{Mpc}\right]+M_{\rm off}.
\]
The upper panel compares the CCDE predictions with the Pantheon Type Ia supernova
sample, while the lower panel shows $m-m_{\Lambda\mathrm{CDM}}$.


In [ ]:

data_obs = np.genfromtxt(
    "Data/SNIa.txt",
    dtype=None,
    names=True,
    encoding=None
)

# ---- Plot style parameters ----
text_size = 30
fig_size_x = 12
fig_size_y = 10
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# ---- Figure: magnitude-redshift relation and difference from LambdaCDM ----
fig = plt.figure(figsize=(fig_size_x, fig_size_y))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)

ax = fig.add_subplot(gs[0])
ax_rat = fig.add_subplot(gs[1], sharex=ax)

# Fixed paper-wide model-to-colour mapping.
colors = [
    '#000000',  # LambdaCDM
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = ["0.0", "0.001", "0.1", "0.001", "-0.1",
                "0.1", "-0.1", "0.1", "0.001", "-0.05"]
sigma_consts = ["0.0", "0.001", "0.3", "0.3", "0.3",
                "1.", "1.", "1.5", "1.5", "1.5"]

COL_Z = 0
COL_dL = 6  # luminosity distance [Mpc]

# Magnitude zero point used in the original analysis.
M_off = 25.0 - 19.453


def load_sorted_m(arr):
    """Return sorted redshift and theoretical apparent magnitude."""
    z = arr[:, COL_Z]
    order = np.argsort(z)

    z = z[order]
    dL = arr[order, COL_dL]

    # D_L=0 at z=0 cannot be used inside log10.
    good = dL > 0
    z = z[good]
    dL = dL[good]

    m_th = 5.0 * np.log10(dL) + M_off
    return z, m_th


# ---- LambdaCDM baseline ----
lcdm_arr = data['bg_data']['sigma=0.0']['alpha=0.0']
z_LCDM, m_LCDM = load_sorted_m(lcdm_arr)

m_LCDM_interp = interp1d(
    np.log1p(z_LCDM),
    m_LCDM,
    kind='linear',
    bounds_error=False,
    fill_value="extrapolate"
)

# ---- CCDE models ----
for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    arr = data['bg_data'].get(sigma_key, {}).get(alpha_key, None)

    if arr is None:
        print("Missing/None:", sigma_key, alpha_key)
        continue

    z_model, m_model = load_sorted_m(arr)

    if sigma_l == "0.0" and alpha_l == "0.0":
        continue

    color = colors[num]
    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'

    ax.plot(
        z_model, m_model, '-',
        lw=lw_f, c=color, label=label
    )

    m_base = m_LCDM_interp(np.log1p(z_model))
    delta_m = m_model - m_base

    ax_rat.plot(
        z_model, delta_m, '-',
        lw=lw_f, c=color
    )

# ---- Pantheon Type Ia supernova data ----
z_data = data_obs["zcmb"]
m_data = data_obs["mb"]
dm_data = data_obs["dmb"]

mask = z_data > 0
z_data = z_data[mask]
m_data = m_data[mask]
dm_data = dm_data[mask]

ax.errorbar(
    z_data,
    m_data,
    yerr=dm_data,
    fmt='.',
    ms=5.5,
    elinewidth=2.0,
    capsize=3,
    color='k',
    mfc='r',
    mec='k',
    alpha=0.5,
    label="SNe Ia (Pantheon)"
)

# ---- Formatting: upper panel ----
ax.set_xscale('log')
ax.set_xlim(0.01, 2.5)
ax.set_ylim(12.5, 30)
ax.set_ylabel(r'$m_B(z)$')
ax.grid(True, which='both', alpha=0.3)
ax.tick_params(direction='in', top=True, right=True)
plt.setp(ax.get_xticklabels(), visible=False)

# ---- Formatting: lower panel ----
ax_rat.set_xscale('log')
ax_rat.set_xlim(0.01, 2.5)
ax_rat.set_xlabel(r'$z$')
ax_rat.set_ylabel(r'$m-m_{\Lambda\mathrm{CDM}}$')
ax_rat.grid(True, which='both', alpha=0.3)
ax_rat.tick_params(direction='in', top=True, right=True)

ax_rat.yaxis.set_major_locator(MultipleLocator(0.1))
ax_rat.yaxis.set_minor_locator(MultipleLocator(0.05))
ax_rat.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

ax.legend(frameon=True, fontsize=16, loc='best', ncol=2)

# plt.tight_layout()

plt.savefig(
    './Figs/m_B_supernova.pdf',
    format='pdf', dpi=300,
    bbox_inches='tight', pad_inches=0.1
)

plt.show()


## 4. CMB temperature angular power spectrum — paper Fig. 8

Load the lensed CMB temperature spectra for the same CCDE parameter choices and compare
them with the Planck 2018 temperature data. The `hi_class` lensed-spectrum files provide
the dimensionless temperature $D_\ell^{TT}$ convention; multiplying by
$T_{\rm CMB}^2$ converts it to $\mu\mathrm{K}^2$.

The upper panel shows $D_\ell^{TT}$ and the lower panel shows
$D_\ell^{TT}/D_{\ell,\Lambda\mathrm{CDM}}^{TT}$, with an inset highlighting the low-$\ell$
region.


In [ ]:
# -------------------------
# Configuration
# -------------------------
redshifts = [100, 50, 30, 20, 10, 6, 5, 4, 3, 2.5, 2, 1.75, 1.5, 1.25, 1.1, 1.0,
             0.9, 0.8, 0.75, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.1, 0.0]

data_address = "./../DataGenerator/cmb/"
load_data = True

# Use strings so directory names match exactly
alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]  


# -------------------------
# Build alpha/sigma arrays
# -------------------------
alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

# Run labels indexed by (sigma, alpha)
out_arr = np.empty((len(sigma), len(alpha)), dtype=object)

# Initialize nested entries
data.setdefault('bg_data', {})
for s in sigma:
    data['bg_data'].setdefault(f'sigma={s}', {})
    for a in alpha:
        data['bg_data'][f'sigma={s}'].setdefault(f'alpha={a}', None)

# -------------------------
# Load LCDM as (alpha=0, sigma=0)
# -------------------------
lcdm_dir = os.path.join(data_address, "LCDM")
lcdm_bg = os.path.join(lcdm_dir, "LCDM_cl_lensed.dat")

data['bg_data'].setdefault('sigma=0.0', {})
data['bg_data']['sigma=0.0'].setdefault('alpha=0.0', None)

if exists(lcdm_bg):
    print("\033[94mLoading LCDM as alpha=0, sigma=0\033[0m")
    data['bg_data']['sigma=0.0']['alpha=0.0'] = np.loadtxt(lcdm_bg)
else:
    print(lcdm_bg,"\033[91mLCDM files not found!\033[0m")

# -------------------------
# Load CCDE runs (alphas × sigmas grid)
# -------------------------
total_loaded = 0
missing = []

if load_data:
    for a_str in alpha:
        for s_str in sigma:
            output_dir = f"sigma{s_str}_alpha{a_str}"
            base_path  = os.path.join(data_address, "run_"+output_dir)
            bg_file    = os.path.join(base_path, "CCDE_"+output_dir+"_cl_lensed.dat")

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i][j] = output_dir

            if exists(bg_file):
                print(f"\033[94m{output_dir} exists — loading\033[0m")
                data['bg_data'][f'sigma={s_str}'][f'alpha={a_str}'] = np.loadtxt(bg_file)
                total_loaded += 1
            else:
                missing.append(bg_file)

print("Number of CCDE simulations loaded:", total_loaded)
print("Alphas loaded:", list(alpha))
print("Sigmas loaded:", list(sigma))
if missing:
    print(f"\033[93mMissing {len(missing)} files (showing up to 10):\033[0m")
    for m in missing[:10]:
        print("  -", m)

# -------------------------------------------------
# Load Planck TT data
# -------------------------------------------------
unbinned = np.loadtxt("Data/TT_full.txt")

ell_u = unbinned[:, 0]
Dl_u = unbinned[:, 1]
dDl_u_neg = unbinned[:, 2]
dDl_u_pos = unbinned[:, 3]


In [ ]:

# ---- Plot style parameters ----
text_size = 30
fig_size_x = 12
fig_size_y = 10
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# ---- Figure: CMB TT spectrum and ratio to LambdaCDM ----
fig = plt.figure(figsize=(fig_size_x, fig_size_y))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)

ax = fig.add_subplot(gs[0])
ax_rat = fig.add_subplot(gs[1], sharex=ax)

# Low-ell inset in the ratio panel.
ax_in = inset_axes(
    ax_rat,
    width="20%",
    height="45%",
    loc="upper left",
    bbox_to_anchor=(0.22, 0.095, 1, 1),
    bbox_transform=ax_rat.transAxes,
    borderpad=0.7
)

ax_in.set_xlim(2, 50)
ax_in.set_ylim(0.95, 1.95)
ax_in.grid(True, which='both', alpha=0.25)
ax_in.tick_params(
    which='both',
    direction='in',
    top=True,
    right=True,
    labelsize=18
)
ax_in.xaxis.set_major_locator(MultipleLocator(10))
ax_in.xaxis.set_minor_locator(MultipleLocator(10))
ax_in.yaxis.set_major_locator(MultipleLocator(0.2))
ax_in.yaxis.set_minor_locator(MultipleLocator(0.1))

mark_inset(
    ax_rat,
    ax_in,
    loc1=2,
    loc2=4,
    fc="none",
    ec="0.2",
    lw=1.2,
    alpha=0.4
)

# Fixed paper-wide model-to-colour mapping.
colors = [
    '#000000',  # LambdaCDM
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

# ---- Planck 2018 TT data ----
ax.errorbar(
    ell_u,
    Dl_u,
    yerr=[dDl_u_neg, dDl_u_pos],
    elinewidth=3.0,
    capsize=4,
    fmt='o',
    ms=4,
    color='k',
    mfc='r',
    mec='k',
    alpha=0.2,
    label="Planck TT"
)

# ---- LambdaCDM baseline ----
lcdm = data['bg_data']['sigma=0.0']['alpha=0.0']

ell_LCDM = lcdm[:, 0]

# The hi_class lensed-cl table stores dimensionless D_ell^{TT}.
# Convert the temperature spectrum to micro-K^2.
Dl_LCDM = lcdm[:, 1] * (2.7255**2) * 1e12

interp_LCDM = interp1d(
    ell_LCDM,
    Dl_LCDM,
    kind='linear',
    bounds_error=False,
    fill_value="extrapolate"
)

# ---- CCDE models ----
for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    if sigma_l == "0.0" and alpha_l == "0.0":
        continue

    arr = data['bg_data'].get(
        f'sigma={sigma_l}', {}
    ).get(
        f'alpha={alpha_l}', None
    )

    if arr is None:
        continue

    ell = arr[:, 0]

    # Dimensionless D_ell^{TT} -> micro-K^2.
    Dl = arr[:, 1] * (2.7255**2) * 1e12

    color = colors[num]
    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'

    ax.plot(
        ell, Dl, '-',
        lw=lw_f, c=color, label=label
    )

    ratio = Dl / interp_LCDM(ell)

    ax_rat.plot(
        ell, ratio, '-',
        lw=lw_f, c=color
    )

    mask_zoom = (ell >= 2) & (ell <= 50)
    ax_in.plot(
        ell[mask_zoom],
        ratio[mask_zoom],
        '-',
        lw=lw_f,
        c=color
    )

ax_in.axhline(1.0, color='k', lw=1.2, alpha=0.8)

# ---- Formatting: upper panel ----
ax.set_ylim(1, 7200)
ax.set_xlim(0, 2000)
ax.set_ylabel(r'$D_\ell^{TT}\ [\mu\mathrm{K}^2]$')
ax.grid(True, which='both', alpha=0.3)
ax.tick_params(direction='in', top=True, right=True)
plt.setp(ax.get_xticklabels(), visible=False)

ax.xaxis.set_major_locator(MultipleLocator(250))
ax.xaxis.set_minor_locator(MultipleLocator(50))

# ---- Formatting: ratio panel ----
ax_rat.set_xlim(0, 2000)
ax_rat.set_ylim(0.45, 2.5)
ax_rat.set_ylabel(r'$D_\ell/D_\ell^{\Lambda\mathrm{CDM}}$')
ax_rat.set_xlabel(r'$\ell$')
ax_rat.axhline(1.0, color='k', lw=1.5)

ax_rat.grid(True, which='both', alpha=0.3)
ax_rat.tick_params(direction='in', top=True, right=True)

ax_rat.yaxis.set_major_locator(MultipleLocator(0.5))
ax_rat.yaxis.set_minor_locator(MultipleLocator(0.1))
ax_rat.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

ax.legend(frameon=True, fontsize=18, loc='best', ncol=2)

# plt.tight_layout()

plt.savefig(
    './Figs/D_ell_Planck.pdf',
    format='pdf', dpi=300,
    bbox_inches='tight', pad_inches=0.1
)

plt.show()
